In [ ]:
import os
#os.environ["CUDA_VISIBLE_DEVICES"] = "0"
#os.environ["CUDA_VISIBLE_DEVICES"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

#os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"



import Analytic_test as AT


import jax
jax.config.update("jax_enable_x64", True)
import numpy as np
import jax.numpy as jnp

import matplotlib.pyplot as plt

In [ ]:
import importlib
importlib.reload(AT)


SphHT = True
integrator = 'leapfrog'
plot = False
dt_override = 20

frozen = True
static = True


l_band_size = 128
use_multi_gpu = True
sparse_k_batch=8192
r_chunk_size = 128
compute_dtype = jnp.complex128

L_out_frac = 1


sim = AT.StellarSimTDep(m22 = 1, r_half = 0.19, r_half_width = 0.05, no_of_particles = 5, no_time_steps = 1000, total_evolve_time = 10, r_min = 20, 
                            r_max_enclosing_frac = 0.99, no_radius_bins = 1000, SphHT = SphHT, integrator = integrator, frozen = frozen, static = static,
                            plot = plot, dt_override=dt_override, l_band_size=l_band_size, use_multi_gpu = use_multi_gpu,
                                sparse_k_batch=sparse_k_batch, r_chunk_size=r_chunk_size, compute_dtype=compute_dtype, L_out_frac=L_out_frac)


In [ ]:

sim.run_simulation()


In [ ]:
positions_all = np.array([p.positions_xyz for p in sim.particles])  # (N_particles, N_steps+1, 3)
r_all         = np.array([p.r_values      for p in sim.particles])  # (N_particles, N_steps+1)
all_vels_cart = np.array([[np.array(v) for v in p.velocities_cart] for p in sim.particles])
kinetic_energy_all = np.array([p.kinetic_energy for p in sim.particles]) # (N_particles, N_steps+1)
potential_energy_all = np.array([p.potential_energy for p in sim.particles]) # (N_particles, N_steps+1)
ang_mom_all = np.array([p.ang_mom for p in sim.particles]) # (N_particles, N_steps+1, 3)


In [ ]:
no_time_steps = sim.no_time_steps
# std across particles (axis=0) at each timestep → shape (n_timesteps,)
stellar_v_disp2 = np.sqrt(
    np.std(all_vels_cart[:, :, 0], axis=0)**2 +
    np.std(all_vels_cart[:, :, 1], axis=0)**2 +
    np.std(all_vels_cart[:, :, 2], axis=0)**2
)


x = np.arange(no_time_steps + 1)  # Time steps array


plt.plot(x * sim.dt * sim.u.to_Gyr, stellar_v_disp2 * sim.u.to_kms)
plt.xlabel('Time [Gyr]')
plt.ylabel('Stellar Velocity Dispersion [km/s]')
plt.title('Stellar Velocity Dispersion over Time')
plt.show()

R_half = np.mean(r_all, axis=0)  # (N_steps+1,) median across particles at each timestep

plt.plot(x * sim.dt * sim.u.to_Gyr, R_half * sim.u.to_Kpc, label='Average Particle Radius')
for particle in range(r_all.shape[0]-1):
    plt.plot(x * sim.dt * sim.u.to_Gyr, r_all[particle, :] * sim.u.to_Kpc, alpha = 0.2, color='gray')
plt.plot(x * sim.dt * sim.u.to_Gyr, r_all[-1, :] * sim.u.to_Kpc, alpha = 0.2, color='gray', label='Individual Particle Radii')
init_r_half = R_half[0] * sim.u.to_Kpc
plt.axhline(init_r_half, color='r', linestyle='--', label='Initial Mean Radius')
plt.xlabel('Time [Gyr]')
plt.ylabel('Average Stellar Radius [Kpc]')
plt.title('Average Stellar Radius over Time')

# Timescale diagnostics
v0 = np.linalg.norm(all_vels_cart[:, 0, :], axis=1)
mean_T_orb = float(np.mean(2 * np.pi * r_all[:, 0] / v0) * sim.u.to_Gyr)

lambda_db_kpc = 19.15 / (sim.m22 * v0 * sim.u.to_kms)
T_c = lambda_db_kpc / (v0 * sim.u.to_Kpc) * sim.u.to_Gyr

#plt.axhline(np.mean(lambda_db_kpc), color='g', linestyle='--', label='$\\lambda_{{\\rm db}}$')


E = np.array(sim.eigen_energies)
freq_diff = np.abs(E[:, None] - E[None, :])
T_beat = (2 * np.pi / freq_diff) * sim.u.to_Gyr
min_T_beat = np.min(T_beat[np.isfinite(T_beat)])
max_T_beat = np.max(T_beat[np.isfinite(T_beat)])

dt_Gyr = sim.dt * sim.u.to_Gyr

info = (
    f"$T_{{\\rm orb}}$ (mean) = {mean_T_orb:.3f} Gyr\n"
    f"$T_{{\\rm c}}$ = {float(np.mean(T_c)):.3f} Gyr\n"
    f"Beat time band: [{min_T_beat:.3f}, {max_T_beat:.3f}] Gyr\n"
    f"$\\Delta t$ = {dt_Gyr:.4f} Gyr"
)

plt.text(1.02, 0.98, info, transform=plt.gca().transAxes,
         verticalalignment='top', fontsize=9,
         bbox=dict(boxstyle='round', facecolor='paleturquoise', alpha=0.6))

plt.legend(bbox_to_anchor=(1, 0.7))
plt.show()


fig, ax = plt.subplots(1, 2, figsize=(12, 5))

for particle in range(ang_mom_all.shape[0]):
    ax[0].plot(x * sim.dt * sim.u.to_Gyr, (kinetic_energy_all[particle] + potential_energy_all[particle]), label='Total Energy')
ax[0].set_xlabel('Time [Gyr]')
ax[0].set_ylabel('Total Energy [J]')
ax[0].set_title('Total Energy over Time')




for particle in range(ang_mom_all.shape[0]):
    ax[1].plot(x * sim.dt * sim.u.to_Gyr, ang_mom_all[particle], label='Total angular momentum')
ax[1].set_xlabel('Time [Gyr]')
ax[1].set_ylabel('Total angular momentum [kg m^2/s]')
ax[1].set_title('Total angular momentum over Time')


plt.tight_layout()
plt.show()
